In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# Summarize Messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="ollama:qwen2.5:3b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:qwen2.5:3b",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking information about Lunapolis, including its climate, population of a specific industry, and potential labor actions.\n\n## SUMMARY\nLunapolis is the capital of the moon. The weather in Lunapolis consists of clear skies with a high temperature of 120°C and a low of -100°C. There are 100,000 cheese miners living in Lunapolis. They have expressed dissatisfaction towards the new president, which may lead to a potential strike.\n\n## ARTIFACTS\n- None\n\n## NEXT STEPS\nNone', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='1e4d9eb2-ac67-4fe5-ad10-a2dd226b26c3'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='c8e89644-a7e3-4bc6-b683-3dcb9b9bf689'),
              AIMessage(content="As Lunapolis's new president, I would

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about Lunapolis, including its climate, population of a specific industry, and potential labor actions.

## SUMMARY
Lunapolis is the capital of the moon. The weather in Lunapolis consists of clear skies with a high temperature of 120°C and a low of -100°C. There are 100,000 cheese miners living in Lunapolis. They have expressed dissatisfaction towards the new president, which may lead to a potential strike.

## ARTIFACTS
- None

## NEXT STEPS
None


# Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:

agent = create_agent(
    model="ollama:qwen2.5:3b",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='a4cd15c7-9ea6-44bc-8850-ccd2ae4451ef'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='7ba4b0cb-bbcb-4a4b-86ae-3c3a198b60ed', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='5a358b10-8e8b-4b03-9024-cae8fca09667'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='e37f9db4-aa9d-4005-b482-b26b7d9d9a65', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='2d773b0e-5d33-4b05-8ba5-a5ff7e6296c4'),
              AIMessage(content="If your device is overheating, it could be a reason why it won

In [10]:
print(response["messages"][-1].content)

If your device is overheating, it could be a reason why it won't turn on. Ensure that the device has enough space around to allow proper heat dissipation.

However, if there are no lights, indicators, or signs of life (like fans running) when you plug in and try to turn on the device, here are some steps you can take:

1. **Power Cycle**: If your device supports it, a hard reset might solve the issue. This involves turning off the power completely and then turning it back on again. 

2. **Check the Power Supply**: Ensure that the power supply is working correctly. You may want to try plugging in another compatible device to see if the problem persists.

3. **Cool Down**: Sometimes devices can be overheated and need some cooling time, especially if you've used them extensively or have been running them in a hot environment. Allow your device to cool down for about 5-10 minutes before trying again.

4. **Revert Recent Changes**: If you made any recent software updates or changes that mig